# Explorer l'IA
Charger des parties, entraîner un petit réseau en regardant la perte, inspecter les embeddings. Lancer d'abord une génération (`loop.py`) ou `ai:selfplay` vers `data/essai`.

In [ ]:
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import numpy as np
import torch

from dataset import Meta, load_samples, sample_files, split_by_game
from model import ValueNet
from train import train, evaluate

## 1. Charger des positions
Choisir un dossier de génération.

In [ ]:
data_dir = sorted(Path('data').glob('*/gen-*'))[-1]  # dernière génération trouvée
meta = Meta.load(data_dir / 'meta.json')
samples = load_samples(sample_files([data_dir]), meta)
train_s, val_s = split_by_game(samples)
print(data_dir, len(train_s), 'positions d’entraînement,', len(val_s), 'de validation')

## 2. Ce que contient une position

In [ ]:
s = samples[0]
print('résultat :', s.y)
for (k, row, n) in s.counts.tolist()[:10]:
    print(meta.set_names[k], 'carte', meta.card_ids[row - 1], '×', n)

## 3. Entraîner sur un petit échantillon
La perte de validation doit baisser, puis remonter quand le réseau apprend par cœur (sur-apprentissage).

In [ ]:
shutil.rmtree('runs/notebook', ignore_errors=True)  # repart de zéro (sinon l'entraînement reprend)
history = []
model, info = train(train_s[:3000], val_s[:500], meta, Path('runs/notebook'), epochs=15, patience=15,
                    log=history.append)
val = [float(line.split('validation ')[1]) for line in history if line.startswith('époque')]
plt.plot(range(1, len(val) + 1), val, marker='o'); plt.xlabel('époque'); plt.ylabel('perte de validation')

## 4. Cartes jugées proches
Distance entre embeddings : des cartes au rôle proche devraient être voisines.

In [ ]:
emb = model.emb.weight.detach().numpy()[1:]  # sans la ligne 0 (inconnue)
norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
similar = norm @ norm.T
card = 0  # indice de la carte étudiée
for k in np.argsort(-similar[card])[1:6]:
    print(meta.card_ids[k], round(float(similar[card, k]), 3))